![Henry Logo](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)


# M3L2 E17b - LM Studio + Qwen (copia práctica)

## Qué es esto

Copia de [E17](../E17_lm_studio_conexion_local/M3L2_E17_Resolution.ipynb) armada específicamente para correr con **Qwen cargado en LM Studio**, sin el resto de la teoría — solo lo necesario para conectar y probar.

### Antes de correr esto

1. Abrí **LM Studio**.
2. En la pestaña de descarga, buscá un modelo **Qwen Instruct** (ej. `qwen2.5-7b-instruct`, `qwen2.5-3b-instruct` si tenés menos RAM, o `qwen3-8b`). Evitá las versiones "base" — necesitás la que dice `Instruct` o `Chat`.
3. Cargalo (`Chat` o `Developer`).
4. En la pestaña **Developer**, click en **Start Server** (puerto por defecto `1234`).

Si algo de esto no está hecho, la celda de conexión de abajo te va a avisar con un mensaje claro en vez de tirar un error críptico.


In [1]:
# Este notebook es independiente: no asume que corriste E17 antes. %pip (no !pip) instala
# en el Python del kernel activo, no depende del PATH de tu shell.
%pip install requests langchain-openai



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Paso 1 — Confirmar que el servidor está arriba y obtener el id real del modelo

LM Studio le pone su propio identificador al modelo que cargaste (no siempre es el nombre "lindo" de la interfaz). Esta celda se lo pregunta directo a la API — así no hay que adivinarlo a mano.


In [3]:
import requests

LMSTUDIO_BASE_URL = "http://localhost:1234/v1"
MODEL_ID = None

try:
    resp = requests.get(f"{LMSTUDIO_BASE_URL}/models", timeout=5)

    if resp.status_code == 401:
        print("Hay algo escuchando en localhost:1234, pero respondio 401 Unauthorized.")
        print("Eso NO es el comportamiento normal de LM Studio (no exige autenticacion).")
        print("Probablemente otro proceso/servicio esta usando ese puerto. Cerralo o")
        print("cambia el puerto del servidor en LM Studio (Developer -> Settings) y")
        print("actualiza LMSTUDIO_BASE_URL arriba.")
    else:
        resp.raise_for_status()
        modelos = [m["id"] for m in resp.json().get("data", [])]

        if not modelos:
            print("El servidor de LM Studio esta arriba, pero no hay ningun modelo cargado.")
            print("Anda a LM Studio -> Chat (o Developer) y carga un modelo Qwen Instruct.")
        else:
            print("LM Studio esta arriba. Modelos disponibles:")
            for m in modelos:
                print(f"  - {m}")
            MODEL_ID = modelos[0]
            print(f"\nUsando automaticamente: {MODEL_ID}")

except requests.exceptions.ConnectionError:
    print("No se pudo conectar a localhost:1234 (nadie esta escuchando en ese puerto).")
    print("Revisa: LM Studio abierto -> pestaña Developer -> Start Server.")
except requests.exceptions.Timeout:
    print("localhost:1234 no respondio a tiempo (timeout).")
    print("El servidor puede estar arrancando todavia, o el modelo se esta cargando.")
except requests.exceptions.RequestException as e:
    print(f"Error de red inesperado hablando con localhost:1234: {e}")

print(f"\nMODEL_ID final: {MODEL_ID}")


LM Studio esta arriba. Modelos disponibles:
  - qwen/qwen3.5-9b
  - text-embedding-nomic-embed-text-v1.5

Usando automaticamente: qwen/qwen3.5-9b

MODEL_ID final: qwen/qwen3.5-9b


> Si `MODEL_ID` quedó en `None`, arreglá lo que te haya avisado arriba y volvé a correr esta celda antes de seguir. Las celdas de abajo dependen de esta.


## Paso 2 — Conectar con LangChain (`ChatOpenAI` apuntando a LM Studio)

In [ ]:
from langchain_openai import ChatOpenAI


llm_qwen = ChatOpenAI(
    model="qwen/qwen3.5-9b",
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",  
    temperature=0.2,
    timeout=60,
    max_retries=1,
)

respuesta = llm_qwen.invoke("Decime solo la palabra: hola")
print(f"Qwen responde: {respuesta.content}")


Qwen responde: 

hola
Tipo de objeto: ChatOpenAI (el mismo ChatOpenAI de siempre, otro base_url)


## Paso 3 — Un prompt un poco más real

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "Sos un profesor de ingenieria de IA. Respondes en una oracion, en español."),
    ("human", "{pregunta}"),
])

chain_qwen = prompt | llm_qwen | StrOutputParser()

print(chain_qwen.invoke({"pregunta": "Que es un servidor de inferencia?"}))


Un servidor de inferencia es una infraestructura computacional optimizada para ejecutar modelos de IA entrenados y desplegar sus predicciones en tiempo real o por lotes, priorizando la latencia baja y el alto rendimiento sobre la capacidad de entrenamiento.


## Paso 4 (opcional) — Tool calling con Qwen

Los modelos Qwen Instruct suelen tener soporte razonable de *tool calling*, pero con modelos chicos (7B-8B) puede fallar más seguido que con GPT-4o-mini: nombre de tool mal formado, argumentos incompletos, o directamente ignora la tool y responde de texto. Si esta celda no detecta ningún `tool_call`, no es un bug tuyo — es una limitación real del modelo/cuantización que estés corriendo.


In [6]:
from langchain_core.tools import tool

@tool
def sumar(a: int, b: int) -> int:
    """Suma dos numeros enteros."""
    return a + b

llm_qwen_con_tools = llm_qwen.bind_tools([sumar])

respuesta_tool = llm_qwen_con_tools.invoke("Cuanto es 234 mas 891? Usa la herramienta si hace falta.")
print(f"Contenido: {respuesta_tool.content!r}")
print(f"Tool calls detectados: {respuesta_tool.tool_calls}")

if respuesta_tool.tool_calls:
    call = respuesta_tool.tool_calls[0]
    resultado = sumar.invoke(call["args"])
    print(f"\nResultado ejecutando la tool a mano: {resultado}")
else:
    print("\nQwen no pidio la tool esta vez (respondio directo). Probá bajar mas la temperatura o reformular la pregunta.")


Contenido: ''
Tool calls detectados: [{'name': 'sumar', 'args': {'a': 234, 'b': 891}, 'id': '789296549', 'type': 'tool_call'}]

Resultado ejecutando la tool a mano: 1125


## Resumen

Esto es exactamente el `model_factory` de [E17](../E17_lm_studio_conexion_local/M3L2_E17_Resolution.ipynb) con `provider="lmstudio"`, pero con el `model` resuelto automáticamente en vez de hardcodeado — así no hace falta ir a buscar el id a mano en `/v1/models` cada vez que cambiás de modelo cargado en LM Studio.
